In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df=spark.read.table(f"adb_caresynce.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")



In [0]:
from pyspark.sql import functions as F

updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()) \
                     .withColumn("update_timestamp", F.current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresynce.silver.{table_name}"

if load_type == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type == "APPEND":
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type == "MERGE":
    if spark.catalog.tableExists(target_table):
        delta_table = DeltaTable.forName(spark, target_table)

        # Build merge condition from comma-separated merge keys
        keys = [k.strip() for k in merge_keys.split(",")]
        merge_condition = " AND ".join(
            [f"target.`{k}` = source.`{k}`" for k in keys]
        )

        # Update all columns except merge keys and insert_timestamp
        update_set = {
            c: F.col(f"source.`{c}`")
            for c in updated_df.columns
            if c not in keys and c != "insert_timestamp"
        }

        # Insert all columns for new records
        insert_values = {c: F.col(f"source.`{c}`") for c in updated_df.columns}

        delta_table.alias("target").merge(
            updated_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(set=update_set) \
         .whenNotMatchedInsert(values=insert_values) \
         .execute()
    else:
        # Target table doesn't exist yet — create it with the first load
        updated_df.write.mode("overwrite").saveAsTable(target_table)

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected FULL, APPEND, or MERGE.")

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))